# Tutorial 0: Onboarding Roadmap

## What is Bayesian metamodeling?

Imagine you have four separate models of early T-cell receptor signaling — one for
membrane geometry, one for CD45 segregation, one for Lck activity, one for TCR
phosphorylation. Each is independently parameterized and reasonably good in its own
domain, but they don't talk to each other. **Bayesian metamodeling is how you make
them agree: you declare which quantities they share, then let each model's
*confidence* — not just its point estimate — decide how much it has to move.** This
curriculum walks you from a 9-point toy sweep to that point in 9 tutorials.

### Why Bayesian, and why not just build one big model?

The obvious alternative is to merge the four into a single ODE system. That fails for
a practical reason: the four models have different representations (a geometry, a
stochastic spatial simulation, a rate law, a reaction network), different owners, and
different calibration data. Merging them demands one shared parameterization nobody
has, and it throws away the calibration each model already earned in its own domain.

Coupling asks for much less. Each model only has to **name a quantity it shares with
another**, and then say how closely the two must agree. Probability is what does the
arbitration: a model that is confident about a shared quantity pulls harder than one
that is vague about it. Hand-tuning shared parameters until the curves overlap also
produces agreement — with no account of who should have yielded, and no way to
propagate what the disagreement cost you.

### One distinction to carry through the whole series

The framework can do two different things with a set of coupled models, and they
answer different questions:

- **Propagation** — "if I take this model's output and push it through the coupling,
  what does that imply downstream?" Information flows one way.
- **Inference** — "what do these models believe once they are *forced* to agree and
  the surrogates' likelihoods are actually evaluated?" Information flows both ways;
  both ends of a coupling move.

`bayesmm meta sample` does **propagation by default** (`--method propagate`) and
inference only when you ask for it (`--method joint`). Only the second produces a
joint posterior, and only the second is what the T-cell paper does. The glossary
below makes this precise; Tutorial 7a shows both, checked against a closed-form
answer.

The framework is **spec-driven** (one typed JSON contract per model), **CLI-first**
(`bayesmm validate / plan / run / surrogate / meta`), and treats every run as a
reproducible experiment with full provenance. The destination of this curriculum is
`projects/tcr_signaling/` — the read-only submodule reproducing Neve-Oz, Sherman &
Raveh (*Frontiers in Immunology*, 2024). You'll meet those four models after T9.

### If you want the idea before the commands

- Raveh et al., *Bayesian metamodeling of complex biological systems across varying
  representations*, PNAS 2021 — the method this framework implements.
  [doi:10.1073/pnas.2104559118](https://doi.org/10.1073/pnas.2104559118)
- Neve-Oz, Sherman & Raveh, *Bayesian metamodeling of early T-cell antigen receptor
  signaling accounts for its nanoscale activation patterns*, Frontiers in Immunology
  2024 — the four coupled models this curriculum builds toward.
  [doi:10.3389/fimmu.2024.1412221](https://doi.org/10.3389/fimmu.2024.1412221)
- `TechSpec.md` in the repo root — the layer architecture and the canonical artifact
  contracts, in a couple of hundred lines.


## What T0 gives you

T0 is orientation, not practice — you won't run a model here. By the end of this
notebook you will be able to:

1. **Say what a metamodel is** in one sentence, and say what a coupling *asserts
   about the world* — not merely how it is spelled in JSON.
2. **Name the five artifacts** that flow through every tutorial: spec → DOE plan →
   sweep → surrogate → metamodel. (The diagram further down is the whole curriculum
   on one line.)
3. **Tell whether your kernel can run a given tutorial** — the final cell probes
   *this* kernel and prints a per-tutorial `ready` / `MISSING` verdict, so you never
   have to guess at environment names.
4. **Decide what to install** — and, just as usefully, what *not* to: the backends
   are per-tutorial, not prerequisites for starting.
5. **Distinguish propagation from inference**, and know which one `bayesmm meta
   sample` hands you by default. That one fact governs how you are entitled to read
   every number T7–T9 print.

If you only remember one thing from T0, make it the data-flow diagram. Every later
tutorial is a zoom-in on one arrow of it.


## What this series is — and what it is not

Worth setting expectations before you invest a day in it, because the gap between the two is
where people get confused.

**What it is: a course in the *principles*, taught on problems small enough to check.**
Almost every model you will meet in these twelve notebooks is a toy — `y = a + b`, a
straight line, a two-variable coupling. That is deliberate. When the answer is available on
paper, you can tell whether the machinery is right, and you spend your attention on the
*idea* rather than on debugging someone else's simulator. Several tutorials check the
framework's output against a closed-form result for exactly this reason.

**What it is not: a reproduction of the published study.** This repository accompanies

> Neve-Oz, Sherman & Raveh, *Bayesian metamodeling of early T-cell antigen receptor
> signaling accounts for its nanoscale activation patterns*, Frontiers in Immunology 15
> (2024), [doi:10.3389/fimmu.2024.1412221](https://doi.org/10.3389/fimmu.2024.1412221)

and the tutorials **do not re-run that paper**. They do not use its super-resolution imaging
data, its Monte-Carlo membrane simulation, or its numbers. What they teach is the *shape* of
what it did, so that when you open the real thing you recognise every part:

| The paper does | You will do, on a toy |
|---|---|
| three partial models built by different people | two or three small models you sweep yourself (T1, T2, 7a) |
| a probabilistic surrogate per model | `pymc_gp` and `sbi_npe` surrogates (T5, T6) |
| coupling variables tying the models together | `gaussian_link` and `deterministic` couplings (7a) |
| inference over the coupled joint with NUTS | `--method joint` and `--method nuts` (7b) |
| infer six hidden parameters from two imaging measurements | infer hidden parameters from two synthetic readouts (7c) |
| finds `Diff = C·P_off` — parameters identifiable only in combination | derive the same kind of ridge, from physics you can check (7c) |

**Where the real thing lives.** `projects/tcr_signaling/` holds the actual partial models and
a metamodel over four of them, with its own notebook series. It is the destination, not the
curriculum — go there once these twelve have made the vocabulary automatic.

**One caveat stated plainly**, because you will notice it: the toy models here are mostly
*linear*, and the surrogate backend used for most of the series (`pymc_gp`) fits linear
models. Real systems are not linear. T5 and T6 are explicit about what that costs you and
how to notice when it bites — that honesty is itself part of the lesson.


## Environment first (simple rule)

Use a Jupyter kernel that already belongs to the conda (or venv) environment you
want. That kernel environment is what persists across all notebook cells.

You don't need to memorize environment names — the preflight cell below runs
**`bayesmm doctor`**, which reports your actual OS, Python, environment, and which
optional backends (PyMC / SBI) are installed. If something is missing, run
**`bayesmm setup`** for platform-correct install commands.

That cell is also your introduction to the two helpers the rest of the series is
built on. `bootstrap()` walks up from the current directory to the repo root (the
`pyproject.toml` that names `bayesian-metamodeling`), chdirs there, and puts `src/`
on `sys.path`. `run_mm_cli(...)` then runs the `bayesmm` CLI **in-process** — no
shell, no `PYTHONPATH=` prefix, and no console script needed on `PATH`, so it
behaves identically on Windows cmd, PowerShell, macOS and Linux. **Tutorials 1–9 all
open with exactly this cell, and none of them explains it again.**

No installation commands run automatically in this notebook.


## Vocabulary you'll see across these tutorials

Definitions are deliberately compact. Later tutorials lean on these terms; if you're confused mid-tutorial, come back here.

### The pipeline

- **Spec**: a typed JSON contract describing one model — its identity, IO schema, runner, adapter, DOE plan, and storage. Validated by `bayesmm validate`.
- **DOE** (*design of experiments*): the set of input points at which you'll run the model. Two strategies in this curriculum: `grid` (cartesian product) and `sobol` (deterministic space-filling).
- **Sweep**: one execution of a model across all DOE points. Produces a centralized `sweep_rows.csv` with one row per point.
- **Adapter**: the small piece of code that turns a DOE point into a process invocation (e.g. `python_cli_adapter_v1`, `biomodels_sbml_adapter_v1`).

### The three Bayesian words the rest of this glossary depends on

- **Prior**: what you believe about a quantity *before* looking at this dataset. In a spec it is the `priors` block — usually a normal with a location and a scale.
- **Likelihood**: how probable the data is under a candidate value of that quantity. This is the job a surrogate does inside a metamodel — and it is exactly why a surrogate must be *probabilistic*. A point predictor offers no likelihood, so there is nothing to condition on and no way to say how far it can be trusted.
- **Posterior**: prior reweighted by likelihood — what you believe *after* the data. The practical test that inference did anything is to compare posterior width against prior width. If the posterior is neither narrower nor shifted, the data told you nothing.

### The metamodeling layer

- **Surrogate**: a fast probabilistic model fit to a sweep's outputs, so you can predict at new inputs without re-running the simulator. Two backends here, and **their ids are not descriptions**:
  - `pymc_gp` — despite the name, **this is not a Gaussian process.** It fits a Bayesian *linear* regression in PyMC: `beta ~ Normal(0, 2)`, `intercept ~ Normal(0, 2)`, `mu = intercept + x @ beta`, `sigma ~ HalfNormal`. No kernel, no covariance function. The id is historical, and renaming it would break every stored artifact and every spec.
  - `sbi_npe` — a neural posterior estimator: a genuinely flexible density model.
  - Why this matters more than a naming quibble: a GP reverts toward its prior mean away from the training data, with error bars that widen to warn you. A Bayesian linear model does the opposite — it extrapolates the fitted plane forever, with narrow bands, far outside the box your DOE covered. On a nonlinear system that is *confidently wrong*, which is more dangerous than visibly uncertain. So judge a fit by held-out error, never by predictive width alone — and notice that this makes **where your DOE put its points** something the surrogate cannot rescue you from. (That is the T4 → T5 link, and it is the reason those two tutorials are adjacent.)
- **Coupling**: a scientific claim, written into the spec, that a variable in model A and a variable in model B are **the same physical quantity** (possibly through a known transform). `deterministic` says they are equal by definition — an identity or a unit conversion, `z = α·C + β`. `gaussian_link` with width σ says they should agree *to within σ*: σ is your honest estimate of how far two independently built models may disagree about the same quantity before you would call it a contradiction. Choosing σ is a modeling decision you have to defend, not a tuning knob. (Older material calls the soft kind `equality_soft`; the notebooks and specs use `gaussian_link`.)
- **Propagation vs. inference**: `bayesmm meta sample` has two modes, and they answer different questions.
  - `--method propagate` (**the default**) draws every variable independently from its prior, then overwrites each coupling's *target* with `transform(source)` (plus noise for a soft link). Information flows one way — the coupling's *source* stays at exactly its prior — and the surrogate likelihoods are never evaluated. This is forward uncertainty propagation. Calling its output a posterior is a misnomer.
  - `--method joint` runs a Metropolis chain over the full joint log-density: priors, couplings **and** surrogate likelihoods. A coupling is then evidence about the *pair*, so it tightens both of its variables. Gradient-free, because a fitted surrogate's `log_prob` is a black box with no gradient to hand a sampler like NUTS.
  - Every stored run records which one ran, in `inference_data.json`'s `method` field: `prior_propagation` or `random_walk_metropolis`. Read that field before you call anything a posterior.
- **Metamodel**: the composed object that wires multiple surrogates together via couplings. Sampling it yields propagated draws — or, under `--method joint`, a joint posterior.
- **Joint posterior**: the multi-variable distribution over all the metamodel's variables *after* conditioning on priors, couplings and surrogate likelihoods — what your coupled models believe **together**, including the correlations the couplings induce. Produced by `--method joint`.
- **Posterior predictive**: the surrogate's belief about the model output at a new input, expressed as a distribution (mean + width) rather than a point estimate. The width is the lesson — subject to the caveat above about which backend produced it.


  There is a third choice, `--method nuts`. It samples the **same** density as `joint` but with gradients, via PyMC, so it is far more efficient and reports proper convergence diagnostics (r-hat, divergences). It only applies when every surrogate is `pymc_gp`, and falls back to `joint` — saying why — otherwise. Tutorial 7b is about it; you do not need it to follow the main series.

In [ ]:
# Cross-platform setup (Windows / macOS / Linux) — no shell, no PYTHONPATH prefix.
# Find the repo root so `src/` is importable, then load the shared tutorial helpers.
# THIS IS THE CELL EVERY LATER TUTORIAL OPENS WITH.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap, run_mm_cli, run_tool  # noqa: F401

root = bootstrap()  # chdir to repo root + ensure src/ on sys.path (idempotent)
ROOT = root
print("Repo root :", root)
print("Python    :", sys.version.split()[0])
print("Kernel env:", sys.prefix)
print()

# CLI preflight. `run_mm_cli` calls the CLI in-process, so a non-zero exit raises
# here — which is the proof that the package is importable in THIS kernel.
run_mm_cli("--version")
print()

# Environment diagnostic. `bayesmm doctor` reports OS, Python, conda/venv, and which
# optional backends (pymc / arviz / torch / sbi) are available. If one is missing,
# `bayesmm setup` prints platform-correct install commands.
_doctor_exit = run_mm_cli("doctor")


## What you need to install (and what's framework vs tutorial)

The framework `bayesian_metamodeling` is intentionally lean. Tutorials add their own infrastructure on top. **Four categories** to keep straight, ordered from "always" to "tutorial-2-only":

| Category | What it is | When you need it | Install (pip) |
|---|---|---|---|
| **Framework runtime** | `pydantic`, `requests`, `scipy` | Always — these come with `pip install -e .` | `pip install -e .` |
| **Optional surrogate backends** | `[pymc]` (T5, T9) and `[sbi]` (T6) | Only for tutorials that *fit* a surrogate | `pip install -e ".[pymc,sbi]"` |
| **Tutorial infrastructure** | `[tutorials]` = `jupyter`, `matplotlib`, `numpy` | To execute the notebooks themselves | `pip install -e ".[tutorials]"` |
| **BioModels SBML simulator** | `[biomodels]` = `libroadrunner`, `tellurium` | **Tutorial 2 only** — to run published BioModels SBML | `pip install -e ".[biomodels]"` |

**The most common gotcha**: `pip install -e ".[pymc,sbi]"` (the README's recommendation) gives you the surrogate backends but NOT `libroadrunner`/`tellurium`. Tutorial 2 will then skip its actual SBML run with a preflight banner. That's fine if you want to skip T2 — install `[biomodels]` only when you're ready to do it.

**One-shot for the impatient**: `pip install -e ".[all]"` installs everything (both backends + tutorial infrastructure + BioModels simulator). Larger download but skips the per-tutorial install dance.

> **About env-name hardcoding in install commands.** Pip's `install` operates on whatever Python it was invoked with — it doesn't care about conda env names. So the commands above are env-agnostic: activate the env running this notebook's kernel, then run them. The package-status cell below prints the kernel env path so you can verify which env is active. The bootstrap cells in T2 (and any tutorial that needs an optional dep) print the same env path before suggesting an install, so the install command always references the right env *for the reader*.

> **About conda for these specific deps.** `pymc` / `arviz` / `pytorch` / `sbi` are all on conda-forge, and the repo ships an env file that solves both backends plus the notebook-execution stack in one go:
>
> ```
> conda env create -f environment-all.yml && conda activate py312_bayesmm_all
> ```
>
> That is the recommended starting point for working through this series. **`environment.yml` is a different thing**: it is the *framework dev* env (`py314_bayesmm`, Python 3.14) and contains **no** `pymc`, `arviz`, `torch` or `sbi`. It is fine for T0, T1, T3, T4, module 7 and T8, and puts T5, T6 and T9 in skip mode. Don't rename either env with `-n` — the `name:` inside the file is what the README, `CLAUDE.md` and the git hooks refer to, and `environment.yml` is pinned to Python 3.14, so a `py312_…` name for it would be actively misleading.
>
> **`libroadrunner` and `tellurium` are PyPI-only — not on conda-forge as of 2026**, so *no* conda env file installs them via conda, including `environment-all.yml`. For Tutorial 2, either create `environment-biomodels.yml` (which pip-installs them inside the conda env) or run `pip install libroadrunner tellurium` in the env you already have.

### Per-tutorial dependency map

| # | Needs `[pymc]` | Needs `[sbi]` | Needs `[biomodels]` | Notes |
|---|:-:|:-:|:-:|---|
| T1 | — | — | — | Pure CLI loop on a toy model |
| T2 | — | — | **YES** | BioModels SBML run; preflight skips cleanly if not installed |
| T3 | — | — | — | Spec validation only — no model execution |
| T4 | — | — | — | DOE planning + the toy run |
| T5 | **YES** | — | — | Fits the `pymc_gp` surrogate (Bayesian linear regression) |
| T6 | — | **YES** | — | Fits the `sbi_npe` surrogate; its Step 4 comparison also wants `[pymc]` |
| **7a** | — | — | — | Runs in any env: `meta sample`'s default `propagate` path is pure NumPy |
| **7b** | YES (Steps 3-6) | — | — | Steps 1-2 need nothing; the sampler comparison and the fitted surrogate need PyMC |
| **7c** | **YES** | optional | — | Conditioning and the cross-model query; Step 5 fits a real `sbi_npe` surrogate if you have it, otherwise substitutes a stand-in and says so |
| T8 | — | — | — | Same — no surrogate is ever loaded, so no backend is ever imported |
| T9 | **YES** | — | — | Capstone — you fit a surrogate yourself |

**Why module 7 and T8 need no backend is not a technicality — it *is* the propagation-vs-inference point.** The default sampler never evaluates a surrogate likelihood, so it never has to load a fitted surrogate, so PyMC is never imported. (module 7's closing section on `--method joint` runs on a hand-built two-variable IR with no surrogates at all, which is why it too runs anywhere.) If a tutorial needs a backend, it is because that tutorial *fits* something.

The package status cell below shows what's installed in *this* kernel, categorized the same way.


In [ ]:
## NOTHING in this cell runs automatically — these are reference commands.
## Pick the line(s) that match what you want to do, paste into a terminal,
## ACTIVATE the env running this notebook's kernel first, then run.
## Restart the Jupyter kernel after installing so cached imports refresh.
##
## ----- Framework runtime (always required; usually already done) -----
#   pip install -e .
##
## ----- Tutorial infrastructure (required to execute these notebooks) -----
#   pip install -e ".[tutorials]"          # jupyter + matplotlib + numpy
##
## ----- Optional surrogate backends (per the per-tutorial map above) -----
#   pip install -e ".[pymc]"               # for T5 and T9 (and T6's comparison step)
#   pip install -e ".[sbi]"                # for T6
##   conda alternatives (these ARE on conda-forge):
#   conda install -c conda-forge pymc arviz
#   conda install -c conda-forge pytorch sbi
##
## ----- Tutorial 2 only: BioModels SBML simulator -----
##   IMPORTANT: libroadrunner and tellurium are PyPI-only, NOT on conda-forge.
##   Use pip even inside a conda env.
#   pip install -e ".[biomodels]"          # libroadrunner + tellurium
#   # or equivalently:
#   pip install libroadrunner tellurium
##
## ----- One-shot: everything for every tutorial -----
#   pip install -e ".[all]"
##
## ----- Recommended for working through the series: the conda env file -----
##   `environment-all.yml` -> env `py312_bayesmm_all`: both surrogate backends
##   (pymc+arviz, pytorch+sbi) plus the notebook-execution stack. It does NOT
##   include libroadrunner/tellurium (PyPI-only), so add them if you want T2.
#   conda env create -f environment-all.yml
#   conda activate py312_bayesmm_all
#   pip install libroadrunner tellurium     # only if you want T2 end-to-end
##
##   `environment.yml` is the FRAMEWORK DEV env (py314_bayesmm, Python 3.14) and
##   ships no surrogate backends. Fine for T0/T1/T3/T4/7a-7c/T8; skip mode elsewhere.
#   conda env create -f environment.yml
#   conda activate py314_bayesmm
##
##   Tutorial 2 has a dedicated env that pip-installs the SBML simulator:
#   conda env create -f environment-biomodels.yml
#   conda activate py312_bayesmm_biomodels
##
## `bayesmm setup` generates a platform-correct version of the above for
## *your* OS + shell. Run it (uncommented) if you want the framework to pick:
#   run_mm_cli("setup")


## Package status (in THIS kernel, categorized)

Reports what's installed in the active Jupyter kernel, grouped by the install matrix above. Anything `not_installed` in a category you need is a clear next step — refer to the install commands in the previous cell.


In [ ]:
import importlib.metadata as md
import importlib.util
import sys


def _check(pkg_name: str, import_name: str | None = None) -> str:
    spec_name = import_name or pkg_name
    if importlib.util.find_spec(spec_name) is None:
        return "not_installed"
    try:
        return md.version(pkg_name)
    except md.PackageNotFoundError:
        return "installed (version unknown)"


# Categorized check matching the install matrix above.
groups = {
    "Framework runtime": [
        ("bayesian-metamodeling", "bayesian_metamodeling"),
        ("pydantic", None),
        ("requests", None),
        ("scipy", None),
    ],
    "Tutorial infrastructure": [
        ("jupyter", None),
        ("nbclient", None),
        ("nbconvert", None),
        ("notebook", None),
        ("matplotlib", None),
        ("numpy", None),
    ],
    "Optional backend: PyMC (needed for T5, T9; T6's comparison step)": [
        ("pymc", None),
        ("arviz", None),
    ],
    "Optional backend: SBI (needed for T6)": [
        ("torch", None),
        ("sbi", None),
    ],
    "Tutorial 2 only: BioModels SBML simulator": [
        ("libroadrunner", "roadrunner"),
        ("tellurium", None),
    ],
}

print(f"Kernel env : {sys.prefix}")
print(f"Python     : {sys.version.split()[0]}")
print()
for group_name, pkgs in groups.items():
    print(f"  {group_name}")
    for pkg_name, import_name in pkgs:
        status = _check(pkg_name, import_name)
        marker = "OK  " if status != "not_installed" else "MISS"
        print(f"    [{marker}] {pkg_name:<25} {status}")
    print()


## Tutorial map

Each tutorial answers one question and leaves you with one new capability. Don't worry about
the time estimates — they're rough.

**Module 7 is three notebooks** (7a, 7b, 7c) rather than one, because coupling is the concept
the framework is named for and it is three distinct questions. 7a needs no optional backend,
so everyone can do the concept; 7b and 7c need PyMC.

| # | What you build | What you'll be able to do |
|---|---|---|
| 1 | Run a 9-point toy sweep, plot two heatmaps from one centralized CSV | Drive the `validate → plan → run` CLI loop end-to-end |
| 2 | Run a real published BioModels SBML model with a dense `k_on` sweep | Recognize the spec contract is the same shape for toy and real models |
| 3 | Break a spec on purpose, then read and fix the validator output | Read a Pydantic validation error and fix the spec without trial-and-error |
| 4 | Compare `grid` vs `sobol` DOE strategies on the same toy | Pick the right DOE strategy for your model's dimensionality |
| 5 | Fit the `pymc_gp` surrogate (Bayesian linear regression, *not* a GP), evaluate it on new inputs with uncertainty | Read a posterior-predictive width — and know why width alone can't tell you a fit is trustworthy |
| 6 | Fit an `sbi_npe` surrogate on the same data, compare it to `pymc_gp` | Tell when the two backends agree, and what disagreement means |
| **7a** | Couple two surrogates with a `gaussian_link` (σ = 0.15) plus a `deterministic` link, and propagate | Write a coupling down as probability, and see why propagation is *not* inference |
| **7b** | Sample the coupled joint properly; compare two samplers against an answer known on paper | Trust a sampler — or catch one that never moved despite a healthy acceptance rate |
| **7c** | Condition on a measurement and infer parameters in *other* models; meet an identifiability ridge | Ask the question the framework exists for: *"I measured this; what does it imply about that?"* |
| 8 | Chain three surrogates, observe uncertainty propagation along the chain | Budget noise across a coupled cascade |
| 9 | Compose your own DOE → sweep → surrogate → evaluation, and write a 3-sentence report | Drive the framework end-to-end through the surrogate layer, unaided — the capstone. Coupling stays in 7a-7c/T8 |

After T9, the natural next step is `projects/tcr_signaling/` — the same workflow at full scale on four real biological models, and the place where `--method joint` runs against genuinely fitted surrogates.


## How the pieces fit together

This is the data flow each tutorial slots into. Tutorials 1-2 cover the left side (spec → run → CSV). Tutorials 3-4 are about specs and DOE planning. Tutorials 5-6 are the middle (CSV → surrogate). Tutorials 7-8 are the right side (surrogates → coupling → metamodel). Tutorial 9 composes the left-to-surrogate stretch unaided.

```
┌────────┐   ┌──────────┐   ┌────────────┐   ┌──────────────────┐   ┌─────────────┐   ┌──────────────┐
│  Spec  │ → │ DOE plan │ → │   Sweep    │ → │ sweep_rows.csv   │ → │  Surrogate  │ → │   Metamodel  │
│ (JSON) │   │  (grid/  │   │ (executes  │   │ (centralized,    │   │  artifact   │   │  + couplings │
│        │   │   sobol) │   │  N points) │   │  one row/point)  │   │  (2 files)  │   │              │
└────────┘   └──────────┘   └────────────┘   └──────────────────┘   └─────────────┘   └──────────────┘
   T3            T4              T1, T2                                  T5, T6               T7, T8
   └───────────────────── T9 composes this stretch, unaided ──────────────────────┘
                                                                                                │
                                                                                                ▼
                                                                                      ┌──────────────────┐
                                                                                      │   Samples over   │
                                                                                      │  all variables   │
                                                                                      │ (samples_dataset │
                                                                                      │     .json)       │
                                                                                      └──────────────────┘
```

**What the bottom box actually contains depends on how you sampled.** Under the default `--method propagate` those draws are forward propagation: every variable from its prior, coupled targets overwritten by `transform(source)`, surrogate likelihoods never evaluated. Under `--method joint` they are a joint posterior. The run's `inference_data.json` records which, in its `method` field.

Three storage artifacts you'll see across the tutorials:

- **`sweep_rows.csv`** — one per sweep, at `<storage.root>/sweeps/<sweep_id>/sweep_rows.csv`. The canonical handoff to surrogates.
- **`artifact.json`** — one per fitted surrogate, under `tmp/surrogate_artifacts/<id>/`. It holds ids, digests, the IO signature and a *path*; the fitted model itself sits beside it in **`backend_payload.json`**. The split is what makes an artifact portable — and it is why `--method joint` can resolve an artifact and still refuse to run: a placeholder artifact with no `backend_payload` has no model to evaluate a likelihood with.
- **`samples_dataset.json`** — one per metamodel sampling run, with `inference_data.json` alongside it recording the method, seed, backend and variables.


## Tips for new lab members

- **Keep a short run log**: commands, run IDs, and a one-line interpretation per major step. The framework's run registry tracks IDs, but your interpretation notes are what make a sweep reproducible six months later.
- **If a dependency is missing, continue with fallback paths.** If `bayesmm doctor` reports SBI missing and you don't plan to do T6, that's fine — skip T6 for now. If `tellurium`/`libroadrunner` are missing, T2 will skip its `bayesmm run` cell with a clear banner — you can still inspect the spec and plan output. Come back when you have time to install. A tutorial in skip mode has demonstrated its plumbing, not its science; note which ones you owe yourself.
- **`MM_BIOMODELS_OFFLINE=1` for sandboxed runs.** On a machine without internet (or when you'd rather not fetch on the spot), set this env var and T2's adapter refuses to download — it uses a cached SBML if you have one, falls back to the copy vendored at `examples/biomodels/`, or prints a `curl` recipe so you can pre-populate the cache from another machine.
- **Inspect one artifact file after each major command.** `cat tmp/run_registry.json | head` after a sweep; `ls tmp/surrogate_artifacts/*/` after a fit (you'll see the `artifact.json` / `backend_payload.json` pair); `cat tmp/metamodel_samples/*/inference_data.json` after a `meta sample` — that last one is where the `method` field lives. Looking at an artifact once teaches you more than reading the docs about it.
- **Time sinks to know about.** T5 and T6's surrogate fits are the only real waits — tens of seconds of PyMC / SBI training. `meta sample` under the default `propagate` is instant (it is NumPy prior draws plus affine transforms); it is `--method joint` that costs seconds to minutes, and module 7's closing section is where you meet it. T2 fetches a published SBML the first time and caches it; after that, and whenever the fetch is blocked, it costs nothing. Everything else is sub-second.
- **The destination is `projects/tcr_signaling/`.** When this curriculum starts feeling like toy work, that's the signal you're ready to read its README.


## Troubleshooting: environment and kernel

Nearly every problem in this curriculum is one of the four below, and all four bite
hardest here in T0 — before you've run anything. The recurring cause is that the
**kernel's** environment is not the environment you installed into.

| Symptom | Cause | Fix |
|---|---|---|
| `ModuleNotFoundError: bayesian_metamodeling` | Package not installed in the kernel's env | `pip install -e .` **inside the env the kernel uses**, then restart the kernel. |
| You installed a backend, but the status cell still says `not_installed` | Installed into a different env than the kernel runs | Compare the `Kernel env path` printed above with where pip reported installing. They must match. This is the single most common problem. |
| `bayesmm: command not found` in a terminal, but the notebook works | The env isn't activated in that terminal | Activate the env you installed into (`conda activate py312_bayesmm_all`, or whatever `bayesmm doctor` reported above). The notebooks never need the console script: they use `run_mm_cli`, which calls the package in-process. |
| Changed the env, notebook still behaves the old way | Kernel still holds the old interpreter | Restart the kernel. Python does not re-read installed packages mid-session. |

**The general move:** when something is missing, don't guess at env names — run
`bayesmm doctor`. It reports the actual interpreter, env and backend status for the
kernel you are really in, which is the thing in dispute.

## Before you move to T1

A 60-second self-check. Questions 1–4 are logistics — T1 assumes all four. Questions
5–7 are the ideas; if you can answer those, the rest of the series will read as
science rather than button-pushing.

1. **What does a spec contain, and what validates it?**
2. **Where does a sweep put its results, and in what format?**
   *(This one matters most: every surrogate in T5–T9 trains from that file.)*
3. **Which tutorials need PyMC, which needs SBI, and which need neither?**
4. **Your kernel right now — can it run T5?** If you don't know, re-read the
   package-status output above rather than finding out mid-T5.
5. **Why must a surrogate be *probabilistic* rather than just a fast fit?**
6. **You write a `gaussian_link` with σ = 0.15 between two models' variables. What
   have you claimed about the world — and what result would send you back to change
   σ?**
7. **You ran `meta sample` with no `--method`. Is the output a posterior?**

**Answers.** (1) Identity, IO schema, runner, adapter, DOE plan and storage;
`bayesmm validate`. (2) `sweep_rows.csv`, one row per DOE point, at
`<storage.root>/sweeps/<sweep_id>/sweep_rows.csv` alongside `sweep_manifest.json` and
`sweep_logs.jsonl`. (3) PyMC → T5, T9 (and T6's comparison step); SBI → T6; neither →
T1, T3, T4, T7, T8. (4) The status cell lists `pymc` as `[OK  ]` with a version or
`[MISS] not_installed`; the final cell turns that into a per-tutorial `ready` /
`MISSING` verdict. (5) Because the metamodel
conditions on its *likelihood*. A point predictor has no likelihood to offer, so
there is nothing to condition on and no way to express how far it can be trusted.
(6) That the two variables are the same physical quantity, to within 0.15 — that a
disagreement smaller than 0.15 is not evidence of a contradiction. If the sampled
residual `y − C` comes out systematically wider than 0.15, the models disagree more
than you allowed, and σ (or one of the models) needs revisiting. (7) No — it is prior
propagation. Confirm it in `inference_data.json`, whose `method` field will read
`prior_propagation` and whose `surrogates_evaluated` field will read `false`.

Onward to **T1**, where you drive the whole loop on a 9-point toy model that runs
in seconds — small enough that you can check every number by hand.


## Final check: what this kernel can actually run

T0's promised deliverable is a **verdict**, so this cell computes one instead of
asserting that a command exited 0.

It checks three things, and each of them can genuinely fail:

1. **Structure** — the environment probe is sound: repo root found and writable, a
   supported Python, all four backend slots reported.
2. **Baseline readiness** — the tutorials T0 claims need no backend really do run
   here. Missing *optional* backends print `MISSING` rather than raising; a partial
   install is a legitimate state, an unusable one is not.
3. **Drift guard** — the two claims this notebook leans on hardest are re-checked
   against the installed source: that `meta sample` still defaults to `propagate`,
   and that `pymc_gp` is still a linear model rather than a Gaussian process. If the
   framework changes, this cell fails and the prose above gets fixed instead of
   quietly rotting. (That is how the errors this notebook once carried survived.)


In [ ]:
# Self-check: a verdict, plus a guard against this notebook's own prose going stale.
import importlib.util as _u
import inspect as _inspect
import re as _re

from bayesian_metamodeling.cli.main import build_parser as _build_parser
from bayesian_metamodeling.config.diagnose import diagnose as _diagnose
from bayesian_metamodeling.surrogates import backends as _backends

# --- (1) structure: the probe itself must be sound -------------------------------
_report = _diagnose()
_bk = _report["backends"]
assert set(_bk) >= {"pymc", "arviz", "torch", "sbi"}, f"unexpected doctor payload: {sorted(_bk)}"
assert _report["repo"]["root"] is not None, (
    "repo root not found — run this notebook from inside the checkout"
)
assert _report["repo"]["writable"], (
    f"repo root {_report['repo']['root']} is not writable; every tutorial writes under tmp/"
)
_pyver = tuple(_report["python"]["version_tuple"][:2])
assert _pyver >= (3, 12), f"Python {_report['python']['version']} is below the 3.12 floor"


# --- (2) readiness verdict: computed, not assumed --------------------------------
def _have(mod: str) -> bool:
    return _u.find_spec(mod) is not None


_base_ok = all(_have(m) for m in ("bayesian_metamodeling", "numpy", "matplotlib"))
_verdict = {
    "T1, T3, T4, T7, T8  (no backend needed)": _base_ok,
    "T5, T9              (pymc + arviz)": _bk["pymc"]["installed"] and _bk["arviz"]["installed"],
    "T6                  (torch + sbi)": _bk["torch"]["installed"] and _bk["sbi"]["installed"],
    "T2                  (libroadrunner)": _have("roadrunner"),
}
assert _base_ok, (
    "the backend-free tutorials cannot run in this kernel: one of "
    "bayesian_metamodeling / numpy / matplotlib is missing"
)

# --- (3) drift guard on the two load-bearing claims above ------------------------
def _find_method_action(parser):
    for action in parser._actions:
        choices = getattr(action, "choices", None)
        if isinstance(choices, dict):
            for sub in choices.values():
                found = _find_method_action(sub)
                if found is not None:
                    return found
        elif "--method" in (action.option_strings or []):
            return action
    return None


_method = _find_method_action(_build_parser())
assert _method is not None, "`meta sample` no longer exposes --method; T0's propagate/joint text is stale"
assert _method.default == "propagate", (
    f"`meta sample --method` now defaults to {_method.default!r}, not 'propagate' — "
    "update the glossary and cell 0 before trusting them"
)
assert set(_method.choices) == {"propagate", "joint", "nuts"}, (
    f"--method choices changed: {_method.choices} — the glossary below lists them, so "
    "add or remove the entry before trusting it"
)

_dispatch = _re.search(r'backend\s*==\s*"pymc_gp"\s*:\s*\n\s*return\s+(\w+)', _inspect.getsource(_backends))
assert _dispatch and _dispatch.group(1) == "_fit_pymc_bayesian_linear", (
    "the pymc_gp backend no longer dispatches to _fit_pymc_bayesian_linear — "
    "re-check the glossary's claim that it is not a Gaussian process"
)
_fit_src = _inspect.getsource(_backends._fit_pymc_bayesian_linear)
assert "pm.math.dot(x, beta)" in _fit_src, "pymc_gp's fitter is no longer a linear mean function"
assert not _re.search(r"pm\.gp[.(]", _fit_src), (
    "pymc_gp now uses pm.gp — it may have become a real Gaussian process, which would "
    "invert this notebook's advice about extrapolation and predictive width"
)

# --- report ----------------------------------------------------------------------
print(f"Kernel     : {_report['env']['kind']} {_report['env']['name'] or ''} -> {_report['python']['executable']}")
print(f"Python     : {_report['python']['version']}   Repo: {_report['repo']['root']} (writable)")
print(f"meta sample: --method defaults to {_method.default!r}, choices {sorted(_method.choices)}")
print()
print("Tutorial readiness in THIS kernel")
print("-" * 52)
for _label, _ok in _verdict.items():
    print(f"  [{'ready  ' if _ok else 'MISSING'}] {_label}")
if not all(_verdict.values()):
    print()
    print("  MISSING is not an error — those tutorials will run in skip mode.")
    print("  `bayesmm setup` prints the install commands for your platform.")
print()
print(f"[T0 self-check OK] kernel: {_u.find_spec('bayesian_metamodeling').origin}")
